### **Create XIC Test File**

Shorten the test file and the results file to add tests for DIA-NN XIC chromatogram loading. Note that the full files are not saved to the repository

In [1]:
import pandas as pd 
import pyarrow.parquet as pq
import pyarrow

#### **Read Data**

In [2]:
parquet = pq.ParquetFile("report_xic/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.xic.parquet")

#### **Get Row Group 1**

In [3]:
parquet_info_0 = parquet.read_row_group(0)

#### **Get Index**

In [4]:
index = parquet.read_row_group(parquet.num_row_groups - 2).to_pandas()
index_0 = index[(index['feature'] == 'index') & (index['info'] == 0)]
index_0 = pyarrow.table(index_0, schema=parquet.schema_arrow)

#### **Get Empty**

In [5]:
empty = parquet_info_0.to_pandas()
empty = pyarrow.table(empty[empty['pr'] == 1], schema=parquet.schema_arrow)

#### **Write Table**

In [6]:
# Create a ParquetWriter
file_path = "output.parquet"
writer = pq.ParquetWriter(file_path, parquet.schema_arrow)

# Write each row group separately
writer.write_table(parquet_info_0)
writer.write_table(index_0)
writer.write_table(empty)

# Close the writer
writer.close()


In [7]:
test = pq.ParquetFile("output.parquet")

In [8]:
test.num_row_groups

3

---

#### **Filter Report**

In [9]:
a = pd.read_parquet("output.parquet")

In [10]:
a

,pr,feature,info,rt,value
0,AAAAAAAAVPSAGPAGPAPTSAAGR2,ms1,0,46.825500,22.001122
1,AAAAAAAAVPSAGPAGPAPTSAAGR2,ms1,0,46.855263,0.000000
2,AAAAAAAAVPSAGPAGPAPTSAAGR2,ms1,0,46.885017,14.000568
3,AAAAAAAAVPSAGPAGPAPTSAAGR2,ms1,0,46.914783,0.000000
4,AAAAAAAAVPSAGPAGPAPTSAAGR2,ms1,0,46.944683,25.001152
...,...,...,...,...,...
60134,AAC(UniMod:4)SSSEEDDC(UniMod:4)VSLSK2,index,0,0.000000,0.000000
60135,AADAEAEVASLNR2,index,0,0.000000,0.000000
60136,AADALEEQQR2,index,0,0.000000,0.000000
60137,AADAVEDLR2,index,0,0.000000,0.000000


In [11]:
report = pd.read_csv("report.tsv", sep='\t')

In [12]:
report[report['Precursor.Id'] == 'AAAAAAAAVPSAGPAGPAPTSAAGR2'].to_csv("report_small.tsv", sep='\t', index=False)